In [ ]:
from datasets import load_dataset

#loading from hugging face
dataset = load_dataset("tattabio/ec_classification")
## making test and train set
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()
##sanity check to make sure dims are the same as on the DGEB paper dimensions
print(train_df.shape)
print(test_df.shape)

In [ ]:
train_df.head()
## checking what the label and seq looks like

In [ ]:
x_train_seq = train_df["Sequence"].tolist()
y_train_labels = train_df["Label"].tolist()

x_test_seq = test_df["Sequence"].tolist()
y_test_labels = test_df["Label"].tolist()

from sklearn.svm import LinearSVC
## defining model
model = LinearSVC()

In [ ]:
!pip install dgeb

In [ ]:
import dgeb
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
## first input which is just the embeddings i am using dgeb embedding model to make each seq into a vector
embedding_model = dgeb.get_model("facebook/esm2_t6_8M_UR50D")

In [ ]:
## converting train data to numerical vectors using last-token pooling
x_train_input1 = embedding_model.encode(x_train_seq)[:, -1, :]

In [ ]:
## doing same for test data
## broke into two cells so we would avoid rerunning

import numpy as np
x_test_p1 = embedding_model.encode(x_test_seq[:127])[:, -1, :]
last_test = embedding_model.encode(x_test_seq[127:])[:, -1, :]
x_test_input1 = np.vstack([x_test_p1, last_test])

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
## checking shape
print(x_train_input1.shape)
print(x_test_input1.shape)
## now fitting model on this info
model.fit(x_train_input1, y_train_labels)
predictions_input1 = model.predict(x_test_input1)

## accuracy and f1 score for input 1
acc1 = accuracy_score(y_test_labels, predictions_input1)
f1_1 = f1_score(y_test_labels, predictions_input1, average="macro")
print("accuracy:", acc1)
print("f1:", f1_1)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, f1_score
## now doing input 2 which is k mer counts of 2

## defining a function to make kmer
def kmerFunction(seq, k):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])


train_kmer2 = [kmerFunction(seq, 2) for seq in x_train_seq]
test_kmer2 = [kmerFunction(seq, 2) for seq in x_test_seq]

## now counting how many times each 2-mer shows up
helper2 = CountVectorizer()
x_train_input2 = helper2.fit_transform(train_kmer2)
x_test_input2 = helper2.transform(test_kmer2)

## checking shape
print(x_train_input2.shape)
print(x_test_input2.shape)

## defining model again
model2 = LinearSVC()
model2.fit(x_train_input2, y_train_labels)
predictions_input2 = model2.predict(x_test_input2)

## printing accuracy and f1 score
acc2 = accuracy_score(y_test_labels, predictions_input2)
f1_2 = f1_score(y_test_labels, predictions_input2, average="macro")
print("accuracy:", acc2)
print("f1:", f1_2)

In [ ]:
## now doing input 3 which is weighted 2-mer
from sklearn.feature_extraction.text import TfidfVectorizer

## defining the weighted helper
weighted2 = TfidfVectorizer()

x_train_input3 = weighted2.fit_transform(train_kmer2)
x_test_input3 = weighted2.transform(test_kmer2)

## ensuring shape is accurate
print(x_train_input3.shape)
print(x_test_input3.shape)

## making model3
model3 = LinearSVC()

model3.fit(x_train_input3, y_train_labels)
predictions_input3 = model3.predict(x_test_input3)

## f1 and accuracy scores
acc3 = accuracy_score(y_test_labels, predictions_input3)
f1_3 = f1_score(y_test_labels, predictions_input3, average="macro")
print("accuracy:", acc3)
print("f1:", f1_3)

In [ ]:
## now doing input 4 which is k mer counts of 3

train_kmer3 = [kmerFunction(seq, 3) for seq in x_train_seq]
test_kmer3 = [kmerFunction(seq, 3) for seq in x_test_seq]

## now counting how many times each 3-mer shows up
helper3 = CountVectorizer()
x_train_input4 = helper3.fit_transform(train_kmer3)
x_test_input4 = helper3.transform(test_kmer3)

## checking shape
print(x_train_input4.shape)
print(x_test_input4.shape)

## defining model again
model4 = LinearSVC()
model4.fit(x_train_input4, y_train_labels)
predictions_input4 = model4.predict(x_test_input4)

## printing accuracy and f1 score
acc4 = accuracy_score(y_test_labels, predictions_input4)
f1_4 = f1_score(y_test_labels, predictions_input4, average="macro")
print("accuracy:", acc4)
print("f1:", f1_4)

In [ ]:
## now doing input 5 which is weighted 3-mer

## defining the weighted helper
weighted3 = TfidfVectorizer()

x_train_input5 = weighted3.fit_transform(train_kmer3)
x_test_input5 = weighted3.transform(test_kmer3)

## ensuring shape is accurate
print(x_train_input5.shape)
print(x_test_input5.shape)

## making model5
model5 = LinearSVC()

model5.fit(x_train_input5, y_train_labels)
predictions_input5 = model5.predict(x_test_input5)

## f1 and accuracy scores
acc5 = accuracy_score(y_test_labels, predictions_input5)
f1_5 = f1_score(y_test_labels, predictions_input5, average="macro")
print("accuracy:", acc5)
print("f1:", f1_5)

In [ ]:
## final results table for SVM across all 5 encodings
import pandas as pd

results = pd.DataFrame({
    "Encoding": [
        "ESM2 embeddings",
        "2-mer counts",
        "2-mer TF-IDF",
        "3-mer counts",
        "3-mer TF-IDF",
    ],
    "Accuracy": [acc1, acc2, acc3, acc4, acc5],
    "F1 (macro)": [f1_1, f1_2, f1_3, f1_4, f1_5],
})

## round for cleaner display
results["Accuracy"] = results["Accuracy"].round(4)
results["F1 (macro)"] = results["F1 (macro)"].round(4)

print("SVM Results")
print("=" * 60)
print(results.to_string(index=False))